In [ ]:
# ==============================================================================
# CÉLULA DE INSTALAÇÃO DEFINITIVA (VERSÕES CONGELADAS)
# Instala as versões exatas de todas as bibliotecas críticas para garantir
# 100% de reprodutibilidade e evitar problemas com futuras atualizações.
# ==============================================================================

!pip install --upgrade --quiet \
"langchain==1.0.0a10" \
"langchain-community==0.3.31" \
"langchain-core==0.4.0.dev0" \
"langchain-google-genai==2.1.12" \
"langchain-openai==0.3.35" \
"langchain-text-splitters==0.3.11" \
"langchain-huggingface==0.3.1" \
"langchain-chroma==0.2.6" \
"transformers==4.57.1" \
"pydantic==2.12.2" \
"pydantic-settings==2.11.0" \
"pydantic_core==2.41.4" \
"accelerate==1.10.1" \
"bitsandbytes==0.48.1" \
"peft==0.17.1" \
"sentence-transformers==5.1.1" \
"faiss-cpu==1.12.0" \
"chromadb==1.2.0" \
"ragas==0.3.7" \
"datasets==4.2.0" \
"tokenizers==0.22.1" \
"huggingface-hub==0.35.3" \
"torch==2.8.0+cu126" \
"torchaudio==2.8.0+cu126" \
"torchvision==0.23.0+cu126" \
"pandas==2.2.2" \
"numpy==1.26.4" \
"scikit-learn==1.6.1" \
"scipy==1.16.2" \
"matplotlib==3.10.0" \
"seaborn==0.13.2" \
"plotly==5.24.1" \
"tqdm==4.67.1" \
"requests==2.32.5"

print("✅ Instalação concluída com sucesso.")
print("   Ambiente reconstruído com as versões exatas de trabalho.")
print("⚠️ ATENÇÃO: REINICIE O AMBIENTE DE EXECUÇÃO AGORA.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 4.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.5/71.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 53.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 468.3/468.3 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.0/76.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.6/460.6 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 374.9/374.9 kB 12.9 MB/s eta 0:

In [ ]:
# @title
# CÉLULA DE INVESTIGAÇÃO AUTOMÁTICA PARA TODAS AS BIBLIOTECAS

import requests
import datetime
from dateutil import parser, tz

# --- 1. LISTA DE BIBLIOTECAS PARA INVESTIGAR ---
# Adicione ou remova bibliotecas conforme necessário para o seu projeto.
LIBRARIES_TO_INVESTIGATE = [
    "transformers",
    "langchain",
    "langchain-core",
    "pydantic",
    "accelerate",
    "bitsandbytes",
    "peft",
    "sentence-transformers"
]

# --- 2. DATA E HORA DE CORTE ---
CUTOFF_DATETIME_STR = "2025-10-17 08:00:00" # Sexta-feira, 8h da manhã

# --- 3. CONFIGURAÇÃO DE FUSO HORÁRIO ---
local_tz = tz.gettz("America/Sao_Paulo")
utc_tz = tz.gettz("UTC")
cutoff_datetime_local = parser.parse(CUTOFF_DATETIME_STR).replace(tzinfo=local_tz)
cutoff_datetime_utc = cutoff_datetime_local.astimezone(utc_tz)

print(f"🔎 Investigando as últimas versões estáveis antes de: {cutoff_datetime_local.strftime('%Y-%m-%d %H:%M:%S %Z%z')}\n")

# --- 4. FUNÇÃO DE INVESTIGAÇÃO ---
def get_latest_version_before_cutoff(package_name, cutoff_utc):
    """Consulta a API do PyPI e retorna a última versão de um pacote antes da data de corte."""
    try:
        url = f"https://pypi.org/pypi/{package_name}/json"
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        data = response.json()

        releases = []
        for version, files in data["releases"].items():
            if files:
                upload_time_str = files[0]["upload_time"]
                naive_datetime = parser.parse(upload_time_str)
                upload_time_utc = naive_datetime.replace(tzinfo=utc_tz)
                releases.append({"version": version, "release_date_utc": upload_time_utc})

        releases.sort(key=lambda r: r["release_date_utc"], reverse=True)

        for release in releases:
            if release["release_date_utc"] < cutoff_utc:
                return release["version"] # Encontrou a versão correta
        return "Não encontrada"
    except requests.exceptions.RequestException:
        return f"Erro ao buscar '{package_name}'"

# --- 5. EXECUÇÃO E APRESENTAÇÃO DOS RESULTADOS ---
final_requirements = []
for library in LIBRARIES_TO_INVESTIGATE:
    latest_version = get_latest_version_before_cutoff(library, cutoff_datetime_utc)
    if latest_version and "Erro" not in latest_version and "Não encontrada" not in latest_version:
        # Usamos '==' para "congelar" na versão exata.
        requirement_line = f'"{library}=={latest_version}"'
        final_requirements.append(requirement_line)
        print(f"✅ {library:<25} -> Última versão estável: {latest_version}")
    else:
        print(f"❌ {library:<25} -> {latest_version}")

print("\n" + "="*80)
print("--- CÉLULA DE INSTALAÇÃO PRONTA PARA COPIAR E COLAR ---")
print("="*80)
print("# CÉLULA DE INSTALAÇÃO (VERSÃO DEFINITIVA - BASEADA NA INVESTIGAÇÃO)")
print("!pip install --upgrade --quiet \\")
# Junta todas as linhas com ' \' no final para o comando pip
pip_install_command = " \\\n    ".join(final_requirements)
print(f"    {pip_install_command}")

# Adiciona as bibliotecas que não precisam de versionamento rígido aqui
print("    # Adicione outras bibliotecas sem versão específica aqui, se necessário")
print("    langchain-community \\")
print("    langchain-google-genai \\")
print("    langchain-huggingface \\")
print("    faiss-cpu \\")
print("    chromadb \\")
print("    langchain-chroma \\")
print("    datasets \\")
print("    ragas")
print("\nprint('\\n✅ Instalação concluída.')")
print("print('⚠️ ATENÇÃO: REINICIE O AMBIENTE DE EXECUÇÃO AGORA.')")
print("="*80)

In [ ]:
# @title
# CÉLULA PARA GERAR E VISUALIZAR TODAS AS VERSÕES (requirements.txt)

import os

print("--- Gerando a lista de todas as bibliotecas e suas versões exatas ---")

# O nome 'requirements.txt' é uma convenção padrão em projetos Python
file_name = "requirements.txt"
# O comando 'pip freeze' lista os pacotes. O '>' salva a saída no arquivo.
!pip freeze > {file_name}

print(f"\n✅ Lista de dependências salva com sucesso no arquivo: '{file_name}'")
print("\n--- Conteúdo do arquivo 'requirements.txt': ---")

# O comando 'cat' exibe o conteúdo do arquivo
!cat {file_name}

In [ ]:
# @title
# CÉLULA DE VERIFICAÇÃO
import transformers
import sentence_transformers
import accelerate

print(f"Versão de Transformers: {transformers.__version__}")
print(f"Versão de Sentence-Transformers: {sentence_transformers.__version__}")
print(f"Versão de Accelerate: {accelerate.__version__}")

try:
    from transformers import PreTrainedModel
    print("\n✅ SUCESSO! 'PreTrainedModel' foi importado corretamente.")
    print("O problema de instalação foi resolvido. Pode continuar executando o resto do seu notebook.")
except ImportError as e:
    print(f"\n❌ FALHA: Ainda não foi possível importar 'PreTrainedModel'. Erro: {e}")
    print("O problema de dependência persiste.")

In [ ]:
# CÉLULA 2: CONFIGURAÇÃO INICIAL E CACHE PERSISTENTE (VERSÃO CORRIGIDA E ROBUSTA)

import os
import glob
from google.colab import userdata
from google.colab import drive
import google.generativeai as genai
from huggingface_hub import login

# 1. Conecta ao Google Drive
print("Conectando ao Google Drive...")
drive.mount('/content/drive')
print("✅ Drive conectado!")

# 2. CONFIGURA O CACHE PERSISTENTE (VERSÃO ROBUSTA)
print("Configurando o cache persistente no Google Drive...")

# --- A MUDANÇA ESTÁ AQUI ---
# Define o caminho base para TODOS os caches
cache_base_path = "/content/drive/MyDrive/LLM-TCC/huggingface_cache"

# Define a variável de ambiente principal
os.environ['HF_HOME'] = cache_base_path

# Define as variáveis específicas para cada sub-biblioteca
# Força os modelos do 'transformers' a salvarem aqui:
os.environ['TRANSFORMERS_CACHE'] = os.path.join(cache_base_path, 'models')
# Força os modelos do 'sentence-transformers' (usado pelo seu embedding) a salvarem aqui:
os.environ['SENTENCE_TRANSFORMERS_HOME'] = os.path.join(cache_base_path, 'sentence_transformers')
# Força os 'datasets' (usados pelo Ragas) a salvarem aqui:
os.environ['HF_DATASETS_CACHE'] = os.path.join(cache_base_path, 'datasets')
# Bônus: Força o 'torch' (PyTorch) a salvar aqui também
os.environ['TORCH_HOME'] = os.path.join(cache_base_path, 'torch')

# Cria as pastas se elas não existirem
os.makedirs(os.environ['TRANSFORMERS_CACHE'], exist_ok=True)
os.makedirs(os.environ['SENTENCE_TRANSFORMERS_HOME'], exist_ok=True)
os.makedirs(os.environ['HF_DATASETS_CACHE'], exist_ok=True)
os.makedirs(os.environ['TORCH_HOME'], exist_ok=True)

print(f"✅ Cache do Hugging Face (HF_HOME) configurado em: {os.environ['HF_HOME']}")
print(f"✅ Cache do Transformers configurado em: {os.environ['TRANSFORMERS_CACHE']}")
print(f"✅ Cache do Sentence-Transformers configurado em: {os.environ['SENTENCE_TRANSFORMERS_HOME']}")
# --- FIM DA MUDANÇA ---

# 3. Configura a chave de API do Google
try:
    api_key = userdata.get('GOOGLE_API_KEY')
    os.environ['GOOGLE_API_KEY'] = api_key
    genai.configure(api_key=api_key)
    print("✅ Chave de API do Google configurada com sucesso.")
except Exception as e:
    print("ERRO: Não foi possível encontrar a GOOGLE_API_KEY.")

# 4. LOGIN NO HUGGING FACE
print("Realizando login no Hugging Face...")
try:
    hf_token = userdata.get('HF_TOKEN')
    login(token=hf_token)
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    print("✅ Sucesso! Login no Hugging Face realizado.")
except Exception as e:
    print("❌ Falha no login. Verifique se o secret 'HF_TOKEN' foi criado corretamente.")

Conectando ao Google Drive...
Mounted at /content/drive
✅ Drive conectado!
Configurando o cache persistente no Google Drive...
✅ Cache do Hugging Face (HF_HOME) configurado em: /content/drive/MyDrive/LLM-TCC/huggingface_cache
✅ Cache do Transformers configurado em: /content/drive/MyDrive/LLM-TCC/huggingface_cache/models
✅ Cache do Sentence-Transformers configurado em: /content/drive/MyDrive/LLM-TCC/huggingface_cache/sentence_transformers
✅ Chave de API do Google configurada com sucesso.
Realizando login no Hugging Face...
✅ Sucesso! Login no Hugging Face realizado.


In [ ]:
# @title
# CÉLULA NOVA: INSTALAÇÃO E CONFIGURAÇÃO DO OLLAMA

print("🚀 Iniciando a configuração do Ollama...")

# 1. Instala o Ollama com um único comando
!curl -fsSL https://ollama.com/install.sh | sh

# 2. Inicia o servidor Ollama em segundo plano.
# O 'nohup' garante que ele continue rodando, e o '&' libera a célula para continuar.
!nohup ollama serve &

# 3. Aguarda alguns segundos para garantir que o servidor esteja totalmente operacional.
import time
print("Servidor Ollama iniciando, aguarde...")
time.sleep(15) # Aumentei o tempo para garantir a inicialização completa
print("Servidor pronto.")

# 4. Baixa o modelo Llama 3 que será usado como juiz.
# (Isso pode levar alguns minutos na primeira vez, pois o modelo tem ~4.7 GB)
# Nas próximas execuções, ele já estará em cache e será instantâneo.
print("Baixando o modelo 'llama3:8b-instruct'... Por favor, aguarde.")
!ollama pull llama3:8b-instruct

print("\n✅ Configuração do Ollama concluída! O Llama 3 está pronto para ser usado como juiz.")

In [ ]:
# @title
# CÉLULA 2.5: DOWNLOAD DOS MODELOS PARA O GOOGLE DRIVE (Alternativa via huggingface-cli)
# Tenta baixar diretamente para o Drive, usando o mesmo local como cache e destino.

import os
import subprocess
from google.colab import userdata

# --- DEFINA OS CAMINHOS LOCAIS (MESMOS DA CÉLULA 4 MODIFICADA) ---
LOCAL_MODEL_PATHS = {
    "llama3": "/content/drive/MyDrive/LLM-TCC/modelos_locais/llama3-8b-instruct",
    "gemma": "/content/drive/MyDrive/LLM-TCC/modelos_locais/gemma-7b-it",
    # Adicione outros se precisar baixar mais modelos
}

# --- MODELOS A BAIXAR (PEGOS DO HUB) ---
MODELS_TO_DOWNLOAD = {
    "llama3": "meta-llama/Meta-Llama-3-8B-Instruct",
    "gemma": "google/gemma-7b-it",
}

# --- TOKEN DO HUGGING FACE ---
hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    print("❌ ERRO: HF_TOKEN não encontrado nos secrets. Não é possível baixar modelos privados/gated.")
else:
    print("✅ Token HF encontrado. Iniciando downloads via huggingface-cli...")

    for model_key, model_id in MODELS_TO_DOWNLOAD.items():
        local_path = LOCAL_MODEL_PATHS.get(model_key)
        if not local_path:
            print(f"⚠️ Aviso: Caminho local não definido para '{model_key}'. Pulando.")
            continue

        print(f"\n--- Verificando/Baixando '{model_id}' para '{local_path}' ---")

        # Verifica se a pasta já existe e tem um arquivo de configuração (indicador de download completo)
        config_file_path = os.path.join(local_path, 'config.json')
        if os.path.exists(config_file_path):
             print(f"✅ Arquivo config.json encontrado em '{local_path}'. Assumindo que o download está completo. Pulando.")
             continue
        else:
            print(f"Pasta '{local_path}' não encontrada ou incompleta. Tentando download...")
            os.makedirs(local_path, exist_ok=True) # Cria a pasta se não existir

            # Constrói o comando CLI
            # Usamos --local-dir e --cache-dir apontando para o MESMO lugar no Drive
            command = [
                "huggingface-cli", "download",
                model_id,
                "--local-dir", local_path,
                "--cache-dir", local_path, # Força o cache a ser no mesmo local de destino
                "--token", hf_token
            ]

            try:
                # Executa o comando e captura a saída
                print(f"Executando comando: {' '.join(command)}")
                process = subprocess.run(command, capture_output=True, text=True, check=True)
                print(f"✅ Download de '{model_id}' concluído com sucesso!")
                # print("Saída do comando:\n", process.stdout) # Descomente para ver a saída detalhada
            except subprocess.CalledProcessError as e:
                print(f"❌ ERRO ao baixar '{model_id}' com huggingface-cli.")
                print(f"   Comando falhou com código de erro: {e.returncode}")
                print(f"   Saída de erro:\n{e.stderr}")
                print(f"   Verifique se aceitou os termos de uso do modelo e se o caminho no Drive está acessível.")
            except Exception as e:
                print(f"❌ ERRO inesperado ao tentar baixar '{model_id}': {e}")

    print("\n🎉 Downloads (ou verificações) via huggingface-cli concluídos.")

In [ ]:
# célula 3
# Funções de Processamento de Texto (Chunking)
import tiktoken
from langchain_core.documents import Document
from tqdm import tqdm

def dividir_texto(texto, max_tokens=500, sobreposicao_tokens=80, modelo="cl100k_base"):
    """Divide o texto em chunks de até `max_tokens`, com `sobreposicao_tokens`."""
    enc = tiktoken.get_encoding(modelo)
    tokens = enc.encode(texto)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk = enc.decode(chunk_tokens)
        chunks.append(chunk)
        if end == len(tokens):
            break
        start += max_tokens - sobreposicao_tokens
    return chunks

def create_documents_from_txt_folder(folder_path):
    """Lê todos os arquivos .txt de uma pasta, extrai metadados e cria os documentos."""
    all_docs = []
    arquivos_txt = glob.glob(f"{folder_path}/*.txt")
    print(f"\nIniciando a criação de chunks a partir de {len(arquivos_txt)} arquivos .txt...")

    for path in tqdm(arquivos_txt, desc="Processando arquivos"):
        with open(path, "r", encoding="utf-8") as f:
            texto = f.read()

        titulo, url = "", ""
        linhas = texto.splitlines()
        for linha in linhas[:5]:
            if linha.startswith("Título:"):
                titulo = linha.replace("Título:", "").strip()
            elif linha.startswith("URL:"):
                url = linha.replace("URL:", "").strip()

        nome_arquivo = os.path.basename(path)
        cd_estrutura = os.path.splitext(nome_arquivo)[0]
        chunks = dividir_texto(texto)

        for i, chunk_content in enumerate(chunks):
            doc = Document(
                page_content=chunk_content,
                metadata={
                    "titulo": titulo, "url": url, "cdEstrutura": cd_estrutura,
                    "arquivo_origem": nome_arquivo, "parte": i + 1, "total_partes": len(chunks)
                }
            )
            all_docs.append(doc)

    print(f"✅ Processamento concluído: {len(all_docs)} documentos (chunks) criados.")
    return all_docs

In [ ]:
# CÉLULA 4 (VERSÃO MODIFICADA PARA CARREGAR DE CAMINHOS LOCAIS NO DRIVE)

import os
import torch
from langchain_community.vectorstores import FAISS
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

# Imports adicionais para carregar modelos do Hugging Face
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
from langchain_community.llms.huggingface_pipeline import HuggingFacePipeline

# --- DEFINA OS CAMINHOS PARA OS MODELOS NO SEU DRIVE ---
# Certifique-se de que essas pastas contêm os arquivos baixados do Hugging Face
# (Use os nomes exatos das pastas onde você salvou os arquivos)
LOCAL_MODEL_PATHS = {
    "llama3": "/content/drive/MyDrive/LLM-TCC/modelos_locais/llama3-8b-instruct",
    "gemma": "/content/drive/MyDrive/LLM-TCC/modelos_locais/gemma-7b-it",
    # Adicione outros modelos se necessário, por exemplo:
    # "mistral": "/content/drive/MyDrive/LLM-TCC/modelos_locais/mistral-7b-instruct",
}

# (As funções carregar_embedding_model e carregar_ou_criar_vector_store permanecem iguais)
def carregar_embedding_model():
    print("Carregando modelo de embedding BAAI/bge-m3...")
    # Se você também quiser carregar o embedding localmente, aplique a mesma lógica:
    # local_embedding_path = "/content/drive/MyDrive/LLM-TCC/modelos_locais/bge-m3"
    # return HuggingFaceEmbeddings(model_name=local_embedding_path, ...)
    return HuggingFaceEmbeddings(
        model_name="BAAI/bge-m3", # Mantendo o embedding do Hub por enquanto
        model_kwargs={'device': 'cuda'},
        encode_kwargs={'normalize_embeddings': True}
    )

def carregar_ou_criar_vector_store(vector_store_nome, db_path, documentos, embedding_model):
    db_path_especifico = f"{db_path}_{vector_store_nome}"
    if os.path.exists(db_path_especifico):
        print(f"Carregando Vector Store '{vector_store_nome}' de: {db_path_especifico}")
        if vector_store_nome == 'faiss':
            return FAISS.load_local(db_path_especifico, embedding_model, allow_dangerous_deserialization=True)
        elif vector_store_nome == 'chroma':
            return Chroma(persist_directory=db_path_especifico, embedding_function=embedding_model)
    else:
        # (O resto da função permanece igual)
        if documentos is None:
             raise FileNotFoundError(f"Banco de dados '{db_path_especifico}' não encontrado. Execute a Célula 5 (Preparação) primeiro.")
        print(f"Criando e salvando novo Vector Store '{vector_store_nome}' em: {db_path_especifico}")
        if vector_store_nome == 'faiss':
             vector_store = FAISS.from_documents(documentos, embedding_model)
             vector_store.save_local(db_path_especifico)
        elif vector_store_nome == 'chroma':
             vector_store = Chroma.from_documents(documents=documentos, embedding=embedding_model, persist_directory=db_path_especifico)
        return vector_store


def carregar_llm(llm_nome):
    print(f"Carregando LLM: {llm_nome}...")
    if 'gemini' in llm_nome: # Gemini continua sendo carregado via API
        # Adiciona um tratamento de erro caso a chave não esteja definida (visto no log anterior)
        google_api_key = userdata.get('GOOGLE_API_KEY')
        if not google_api_key:
            print("❌ ERRO: GOOGLE_API_KEY não encontrada nos secrets. Gemini não pode ser carregado.")
            raise ValueError("GOOGLE_API_KEY não encontrada.")
        return ChatGoogleGenerativeAI(
            model=llm_nome,
            google_api_key=google_api_key,
            temperature=0.3,
            request_timeout=240
        )
    else:
        # --- MUDANÇA PRINCIPAL: USA O CAMINHO LOCAL ---
        if llm_nome not in LOCAL_MODEL_PATHS:
            raise ValueError(f"Caminho local para o modelo '{llm_nome}' não definido em LOCAL_MODEL_PATHS.")

        model_path = LOCAL_MODEL_PATHS[llm_nome] # Pega o caminho do Drive

        if not os.path.isdir(model_path):
             raise FileNotFoundError(f"Diretório do modelo não encontrado em: {model_path}. Verifique o caminho e se os arquivos foram baixados corretamente para esta pasta.")

        print(f"Carregando modelo de '{model_path}' (Google Drive) com quantização de 4-bit...")
        # -------------------------------------------------

        hf_token = userdata.get('HF_TOKEN') # Token ainda pode ser útil para configurações

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        # Carrega o tokenizer DO CAMINHO LOCAL
        # O use_fast=False para llama3 AINDA É NECESSÁRIO porque depende dos *arquivos* do tokenizer
        # que foram baixados (a versão lenta vs rápida dos arquivos de configuração/vocabulário)
        if llm_nome == "llama3":
             print("Aplicando correção: Carregando tokenizer LOCAL do Llama3 com 'use_fast=False'.")
             tokenizer = AutoTokenizer.from_pretrained(
                 model_path, # <--- Usa caminho local
                 token=hf_token,
                 use_fast=False # <--- Mantém a correção
             )
        else:
            # Gemma e outros carregam normalmente do caminho local
            tokenizer = AutoTokenizer.from_pretrained(model_path, token=hf_token) # <--- Usa caminho local


        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token
            print("Aviso: 'pad_token' não encontrado. Definido como 'eos_token'.")

        # Carrega o modelo DO CAMINHO LOCAL
        model = AutoModelForCausalLM.from_pretrained(
            model_path, # <--- USA O CAMINHO LOCAL
            quantization_config=bnb_config,
            device_map="auto",
            token=hf_token # Pode ser necessário para configs mesmo localmente
        )

        pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tokenizer,
            max_new_tokens=1024,
            temperature=0.2
        )

        return HuggingFacePipeline(pipeline=pipe)


print("✅ Funções de carregamento (versão para CAMINHOS LOCAIS NO DRIVE) prontas.")

/usr/local/lib/python3.12/dist-packages/transformers/utils/hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


✅ Funções de carregamento (versão para CAMINHOS LOCAIS NO DRIVE) prontas.


In [ ]:
# CÉLULA 5: PREPARAÇÃO E INDEXAÇÃO (VERSÃO FINAL COM TODAS AS CORREÇÕES)

import pickle
import os
import glob
import tiktoken
# CORREÇÃO 1: Importa Loaders do langchain_community
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
# CORREÇÃO 2: Importa Document do langchain_core
from langchain_core.documents import Document
from tqdm import tqdm
# CORREÇÃO 3: Importar FAISS e Chroma (assumindo que estão em células anteriores)
# Se não estiverem, descomente as linhas abaixo
# from langchain_community.vectorstores import FAISS
# from langchain_community.vectorstores import Chroma
# import shutil # Para o Chroma

print("--- INICIANDO ETAPA DE PREPARAÇÃO E CRIAÇÃO DOS BANCOS DE DADOS VETORIAIS ---")

# --- 1. Definição dos Caminhos e Parâmetros ---
base_path = "/content/drive/MyDrive/LLM-TCC/"
vectorstore_base_path = base_path + "dbs"
PASTA_TXT = base_path + "textos_comdescricao"
TIPOS_DE_VECTOR_STORE = ['faiss', 'chroma']
documentos_cache_path = base_path + "documentos_processados.pkl"

# --- NOVA FUNÇÃO DE CHUNKING (MESCLADA) ---

def dividir_texto_com_tiktoken(texto, max_tokens=500, sobreposicao_tokens=80, modelo="cl100k_base"):
    """Divide o texto em chunks de até `max_tokens`, com `sobreposicao_tokens`."""
    enc = tiktoken.get_encoding(modelo)
    tokens = enc.encode(texto)
    chunks = []
    start = 0
    while start < len(tokens):
        end = min(start + max_tokens, len(tokens))
        chunk_tokens = tokens[start:end]
        chunk = enc.decode(chunk_tokens)
        chunks.append(chunk)
        if end == len(tokens):
            break
        start += max_tokens - sobreposicao_tokens
    return chunks

# --- FUNÇÃO DE CHUNKING MODIFICADA (IMPLEMENTANDO SUA LÓGICA) ---
def criar_e_identificar_chunks_completo(caminho_pasta_txt):
    """
    Carrega arquivos .txt, extrai metadados, LIMPA o texto, divide em chunks,
    e prefixa CADA chunk com os metadados antes de criar o Documento.
    Atribui um ID único e rastreável.
    """
    print(f"Lendo arquivos de texto de: {caminho_pasta_txt}")
    arquivos_txt = glob.glob(f"{caminho_pasta_txt}/*.txt")
    print(f"Encontrados {len(arquivos_txt)} arquivos .txt.")

    todos_os_chunks = []

    for path in tqdm(arquivos_txt, desc="Processando arquivos e criando chunks"):
        with open(path, "r", encoding="utf-8") as f:
            texto_completo = f.read()

        # 1. EXTRAÇÃO DE METADADOS
        titulo, url = "", ""
        linhas = texto_completo.splitlines()
        linhas_de_cabecalho = 0 # Contador para saber quantas linhas pular

        # Procura o cabeçalho nas primeiras 5 linhas
        for i, linha in enumerate(linhas[:5]):
            if linha.startswith("Título:"):
                titulo = linha.replace("Título:", "").strip()
                linhas_de_cabecalho = max(linhas_de_cabecalho, i + 1)
            elif linha.startswith("URL:"):
                url = linha.replace("URL:", "").strip()
                linhas_de_cabecalho = max(linhas_de_cabecalho, i + 1)

        nome_arquivo = os.path.basename(path)
        cd_estrutura = os.path.splitext(nome_arquivo)[0]

        # 2. LIMPEZA DO TEXTO
        # Remove as linhas de cabeçalho que já lemos (ex: Título, URL)
        # para que o texto a ser dividido em chunks esteja "limpo"
        texto_limpo = "\n".join(linhas[linhas_de_cabecalho:])

        # 3. DIVISÃO EM CHUNKS (usando o texto limpo)
        chunks_de_texto_limpo = dividir_texto_com_tiktoken(texto_limpo)

        # 4. CRIAÇÃO DE DOCUMENTOS (Prefixando o cabeçalho em TODOS)

        # Cria o cabeçalho padronizado para ser adicionado a CADA chunk
        # Isso garante consistência para o embedding (bge-m3)
        cabecalho_padronizado = ""
        if titulo:
            cabecalho_padronizado += f"Título: {titulo}\n"
        if url:
            cabecalho_padronizado += f"URL: {url}\n"
        # Adiciona uma separação se o cabeçalho existir
        if cabecalho_padronizado:
            cabecalho_padronizado += "\n"

        for i, chunk_content in enumerate(chunks_de_texto_limpo):

            # IMPLEMENTAÇÃO DA LÓGICA:
            # Concatena o cabeçalho padronizado + o conteúdo do chunk limpo
            novo_page_content = cabecalho_padronizado + chunk_content

            novo_chunk = Document(
                page_content=novo_page_content, # <-- VERSÃO NOVA E CONSISTENTE
                metadata={
                    "titulo": titulo, # Metadado original ainda é salvo
                    "url": url,
                    "cdEstrutura": cd_estrutura,
                    "arquivo_origem": nome_arquivo,
                    "parte": i + 1,
                    "total_partes": len(chunks_de_texto_limpo),
                    'id': f'{nome_arquivo}_chunk_{i}'
                }
            )
            todos_os_chunks.append(novo_chunk)

    print(f"Total de {len(todos_os_chunks)} chunks criados com IDs únicos e cabeçalhos consistentes.")
    return todos_os_chunks

# --- 2. Lógica de Carregar ou Criar os Documentos (Chunks) ---
# (O resto do código permanece o mesmo)
if os.path.exists(documentos_cache_path):
    print(f"Carregando documentos (chunks) do cache: {documentos_cache_path}")
    with open(documentos_cache_path, "rb") as f:
        documentos = pickle.load(f)
else:
    print("Cache de documentos não encontrado. Executando o processo de chunking e identificação...")
    documentos = criar_e_identificar_chunks_completo(PASTA_TXT)

    print("Salvando documentos processados (com IDs) em cache para uso futuro...")
    with open(documentos_cache_path, "wb") as f:
        pickle.dump(documentos, f)

print(f"✅ {len(documentos)} documentos (chunks) carregados.")
print("\nExemplo de metadados do primeiro chunk:")
# Imprime o primeiro chunk se ele existir
if documentos:
    print(documentos[0].metadata)
    print("\nExemplo de CONTEÚDO do primeiro chunk (como o embedding verá):")
    print(documentos[0].page_content[:300] + "...") # Imprime os primeiros 300 chars
else:
    print("Nenhum documento foi carregado ou criado.")


# --- 3. Carregamento do Modelo de Embedding ---
# Assume que a função carregar_embedding_model() está definida em outra célula
embedding_model = carregar_embedding_model()

# --- 4. Loop para Recriar cada Banco de Dados Vetorial ---
print("\n--- Recriando bancos de dados vetoriais com os novos IDs ---")
for tipo_db in TIPOS_DE_VECTOR_STORE:
    print("-" * 50)
    db_path_especifico = f"{vectorstore_base_path}_{tipo_db}"
    print(f"Forçando a recriação do Vector Store '{tipo_db}' em: {db_path_especifico}")

    if tipo_db == 'faiss':
        # Assume que FAISS foi importado
        vector_store = FAISS.from_documents(documentos, embedding_model)
        vector_store.save_local(db_path_especifico)
    elif tipo_db == 'chroma':
        # Assume que Chroma e shutil foram importados
        import shutil
        if os.path.exists(db_path_especifico):
            shutil.rmtree(db_path_especifico)
        vector_store = Chroma.from_documents(documents=documentos, embedding=embedding_model, persist_directory=db_path_especifico)

    print(f"✅ Vector Store '{tipo_db}' recriado com sucesso.")

print("\n✅ PREPARAÇÃO CONCLUÍDA! Seus bancos de dados vetoriais foram recriados com chunks identificáveis e consistentes.")

--- INICIANDO ETAPA DE PREPARAÇÃO E CRIAÇÃO DOS BANCOS DE DADOS VETORIAIS ---
Cache de documentos não encontrado. Executando o processo de chunking e identificação...
Lendo arquivos de texto de: /content/drive/MyDrive/LLM-TCC/textos_comdescricao
Encontrados 927 arquivos .txt.


Processando arquivos e criando chunks: 100%|██████████| 927/927 [00:33<00:00, 27.53it/s] 


Total de 2314 chunks criados com IDs únicos e cabeçalhos consistentes.
Salvando documentos processados (com IDs) em cache para uso futuro...
✅ 2314 documentos (chunks) carregados.

Exemplo de metadados do primeiro chunk:
{'titulo': 'TOMATE - Calagem e Adubação - Adubação - Recomendação no SISPIT (EE-Caçador)', 'url': 'https://sistemas.epagri.sc.gov.br/sedimob/consulta.action?subFuncao=consultaDiagnostico&cdEstrutura=024&isEdicao=N&epagriTEC=S', 'cdEstrutura': '024', 'arquivo_origem': '024.txt', 'parte': 1, 'total_partes': 4, 'id': '024.txt_chunk_0'}

Exemplo de CONTEÚDO do primeiro chunk (como o embedding verá):
Título: TOMATE - Calagem e Adubação - Adubação - Recomendação no SISPIT (EE-Caçador)
URL: https://sistemas.epagri.sc.gov.br/sedimob/consulta.action?subFuncao=consultaDiagnostico&cdEstrutura=024&isEdicao=N&epagriTEC=S


[IMAGEM] http://intranetdoc.epagri.sc.gov.br/sedimob/img/sedimob_icone_tomate_col...
Carregando modelo de embedding BAAI/bge-m3...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]


--- Recriando bancos de dados vetoriais com os novos IDs ---
--------------------------------------------------
Forçando a recriação do Vector Store 'faiss' em: /content/drive/MyDrive/LLM-TCC/dbs_faiss
✅ Vector Store 'faiss' recriado com sucesso.
--------------------------------------------------
Forçando a recriação do Vector Store 'chroma' em: /content/drive/MyDrive/LLM-TCC/dbs_chroma
✅ Vector Store 'chroma' recriado com sucesso.

✅ PREPARAÇÃO CONCLUÍDA! Seus bancos de dados vetoriais foram recriados com chunks identificáveis e consistentes.


In [ ]:
# @title
######     quais geminis posso usar?


import google.generativeai as genai
from google.colab import userdata

# Configura a chave de API (certifique-se de que o secret 'GOOGLE_API_KEY' existe)
try:
    api_key = userdata.get('GOOGLE_API_KEY')
    genai.configure(api_key=api_key)

    print("✅ Chave de API configurada. Listando modelos disponíveis para 'generateContent':\n")

    # Lista os modelos e verifica se eles suportam o método 'generateContent'
    for model in genai.list_models():
        if 'generateContent' in model.supported_generation_methods:
            print(f"- {model.name}")

except Exception as e:
    print(f"ERRO: Não foi possível obter a chave ou listar os modelos. Detalhes: {e}")

In [ ]:
# CÉLULA 6AAAAAAA (VERSÃO FINAL: THRESHOLD L2 + FALLBACK + COMENTÁRIOS)

import pandas as pd
import os
import torch
from tqdm import tqdm
from sentence_transformers import CrossEncoder
import json
import math # Para a nota explicativa sobre a conversão

# --- 1. INSTALAÇÃO (Comente/Delete se já feita na Célula 1) ---
# !pip install -q sentence-transformers

# --- 2. DEFINIÇÃO DOS THRESHOLDS ---
# Threshold da Distância L2 do FAISS para o "caminho rápido" (MENOR é MELHOR).
# Escolhemos 0.4 como ponto de partida.
# NOTA: Como os embeddings bge-m3 são normalizados (comprimento 1), existe uma relação
# matemática direta entre Distância L2 (d) e Similaridade de Cosseno (s): s = 1 - (d^2 / 2).
# Portanto, uma Distância L2 de 0.4 corresponde a uma Similaridade de Cosseno de:
# s = 1 - (0.4^2 / 2) = 1 - (0.16 / 2) = 1 - 0.08 = 0.92.
#similaridade to distance
# 1.00 - 0.000 Identicos
# 0.99 - 0.141 Extremamente Similares
# 0.95 - 0.316 Muito Similares
# 0.92 - 0.400 (Threshold L2)
# 0.90 - 0.447 Bastante Similares
# 0.85 - 0.548 Similares
# 0.80 - 0.632 (Threshold Cosseno)
# 0.75 - 0.707 Moderadamente Similares
# 0.70 - 0.775 Razoavelmente Similares
# 0.60 - 0.894 Alguma Similaridade
# 0.50 - 1.000 (Limite Re-ranker)
# 0.30 - 1.183 Pouco Similares
# 0.00 - 1.414 (Não Relacionados (Ortogonais)
# -1.00 - 2.000Opostos
THRESHOLD_RAPIDO_L2 = 0.45

# Threshold do Re-ranker (MAIOR é MELHOR, de 0 a 1). 0.5 é um ponto de partida comum.
THRESHOLD_RERANKER = 0.7
K_INICIAL_RERANK = 20 # Quantos candidatos pegar para o re-ranker se o caminho rápido falhar
K_FINAL = 5 # Quantos chunks salvar/enviar ao LLM

# --- 3. PREPARAÇÃO DOS CAMINHOS E DADOS ---
base_path = "/content/drive/MyDrive/LLM-TCC/"
vectorstore_base_path = base_path + "dbs"
# Nome do arquivo de saída desta célula
contextos_salvos_path = os.path.join(base_path, "contextos_recuperados_com_decisao_L2_fallback.csv")

golden_dataset_path = os.path.join(base_path, "golden_dataset_tomate.csv")
golden_dataset = pd.read_csv(golden_dataset_path)
print(f"✅ Golden dataset carregado com {len(golden_dataset)} perguntas.")

# --- 4. CARREGAMENTO DOS MODELOS DE RECUPERAÇÃO ---
print("Carregando modelo de Embedding (para FAISS)...")
# Assume que carregar_embedding_model() usa device='cuda'
embedding_model = carregar_embedding_model()

print("Carregando Vector Store (FAISS)...")
config_vector_store = 'chroma'
recuperador = carregar_ou_criar_vector_store(
    config_vector_store,
    vectorstore_base_path,
    None,
    embedding_model
)
print("✅ Vector Store carregado.")

print("Descarregando modelo de Embedding da GPU...")
del embedding_model
torch.cuda.empty_cache()
print("✅ Modelo de Embedding descarregado.")

print("Carregando modelo de Re-ranking (BAAI/bge-reranker-large) na GPU...")
reranker = CrossEncoder('BAAI/bge-reranker-large', device='cuda')
print("✅ Modelo de Re-ranking carregado.")

# --- 5. EXECUÇÃO DO LOOP DE RECUPERAÇÃO COM DECISÃO (USANDO L2) ---
resultados_recuperacao = []

for index, row in tqdm(golden_dataset.iterrows(), total=len(golden_dataset), desc="Processando recuperação e re-ranking"):
    pergunta = row['question']
    documentos_finais = []
    status_recuperacao = ""
    score_inicial_l2 = -1.0 # Usamos -1.0 para indicar que não foi calculado ainda
    scores_rerank = []

    # ETAPA 1: Recuperação Rápida (Top 1 com score L2)
    docs_com_score_k1 = recuperador.similarity_search_with_score(pergunta, k=1)

    documentos_candidatos = [] # Define a variável aqui para o escopo

    if not docs_com_score_k1:
        status_recuperacao = "Falha_Nenhum_Resultado_Inicial"
        # Mesmo falhando aqui, tentamos buscar candidatos para o fallback
        documentos_candidatos = recuperador.similarity_search(pergunta, k=K_INICIAL_RERANK)
        if not documentos_candidatos:
             status_recuperacao = "Falha_Total_Sem_Candidatos"
    else:
        melhor_doc_inicial, score_inicial_l2 = docs_com_score_k1[0]

        # ETAPA 2: DECISÃO INICIAL (Baseada em L2)
        if score_inicial_l2 <= THRESHOLD_RAPIDO_L2:
            status_recuperacao = "Sucesso_Caminho_Rapido"
            # Pega os Top 5 direto do FAISS (usando similarity_search que é o correto)
            documentos_finais = recuperador.similarity_search(pergunta, k=K_FINAL)
        else:
            # A busca rápida não foi confiável (L2 muito alta), precisa escalar
             documentos_candidatos = recuperador.similarity_search(pergunta, k=K_INICIAL_RERANK)
             if not documentos_candidatos:
                  status_recuperacao = "Falha_Rerank_Sem_Candidatos"

    # --- Bloco de Re-ranking (Executado se status ainda NÃO for Sucesso_Caminho_Rapido ou Falha_Total) ---
    if status_recuperacao not in ["Sucesso_Caminho_Rapido", "Falha_Total_Sem_Candidatos"]:
        # Garante que temos candidatos, mesmo se a busca inicial falhou
        if not documentos_candidatos:
             documentos_candidatos = recuperador.similarity_search(pergunta, k=K_INICIAL_RERANK)
             if not documentos_candidatos: # Checagem dupla
                  status_recuperacao = "Falha_Total_Sem_Candidatos_Apos_Rerank"
                  # Define listas vazias para evitar erros adiante
                  documentos_finais = []
                  scores_rerank = []

        # Somente executa o reranker se TIVER candidatos e NÃO for Falha_Total
        if status_recuperacao != "Falha_Total_Sem_Candidatos_Apos_Rerank":
            # ETAPA 3: Escalada (Re-rank)
            pares = [(pergunta, doc.page_content) for doc in documentos_candidatos]
            scores = reranker.predict(pares)
            docs_com_scores = list(zip(scores, documentos_candidatos))

            # Ordena ANTES de filtrar para usar no fallback
            docs_com_scores.sort(key=lambda x: x[0], reverse=True)

            # ETAPA 4: Filtro de Qualidade do Re-ranker
            documentos_filtrados = [(s, d) for s, d in docs_com_scores if s >= THRESHOLD_RERANKER]

            # ETAPA 5: DECISÃO FINAL MODIFICADA (com Fallback)
            if not documentos_filtrados:
                # --- LÓGICA DE FALLBACK ---
                status_recuperacao = "Alerta_Baixa_Confianca"
                # Pega os Top 5 do re-ranker, mesmo com score baixo
                documentos_finais_tuplas = docs_com_scores[:K_FINAL]
                documentos_finais = [d for s, d in documentos_finais_tuplas]
                scores_rerank = [float(s) for s, d in documentos_finais_tuplas]
                # --- FIM DO FALLBACK ---
            else:
                status_recuperacao = "Sucesso_Rerank"
                documentos_finais_tuplas = documentos_filtrados[:K_FINAL] # Pega top K dos filtrados
                documentos_finais = [d for s, d in documentos_finais_tuplas]
                scores_rerank = [float(s) for s, d in documentos_finais_tuplas]

    # --- Preparação para Salvar ---
    # Garante que documentos_finais é sempre uma lista, mesmo em falhas
    if status_recuperacao.startswith("Falha_"):
        documentos_finais = []
        scores_rerank = []

    contexto_lista_ids = [doc.metadata.get('id', None) for doc in documentos_finais]
    contexto_lista_textos = [doc.page_content for doc in documentos_finais]

    # Salva o resultado da recuperação
    resultados_recuperacao.append({
        'question': pergunta,
        'ground_truth': row['ground_truth'],
        'status_recuperacao': status_recuperacao,
        'faiss_top1_score(L2)': float(score_inicial_l2) if score_inicial_l2 != -1.0 else None,
        # Adiciona a nota de Cosseno apenas como informação extra
        'faiss_top1_score(Cosine_Est)': (1 - (min(score_inicial_l2, math.sqrt(2))**2 / 2)) if score_inicial_l2 != -1.0 else None,
        'reranker_top_scores': scores_rerank,
        'context_ids': contexto_lista_ids,
        'context_texts': contexto_lista_textos
    })

# --- 6. SALVAMENTO DO CSV DE CONTEXTOS ---
df_contextos = pd.DataFrame(resultados_recuperacao)

# Converte listas em strings JSON
df_contextos['reranker_top_scores'] = df_contextos['reranker_top_scores'].apply(json.dumps)
df_contextos['context_ids'] = df_contextos['context_ids'].apply(json.dumps)
df_contextos['context_texts'] = df_contextos['context_texts'].apply(json.dumps)

df_contextos.to_csv(contextos_salvos_path, index=False)

print(f"\n🎉 Processo de recuperação e re-ranking concluído! Os contextos foram salvos em:\n{contextos_salvos_path}")
print("   Verifique a coluna 'status_recuperacao' para a lógica de decisão.")

# Limpa a memória
del recuperador
del reranker
torch.cuda.empty_cache()

✅ Golden dataset carregado com 10 perguntas.
Carregando modelo de Embedding (para FAISS)...
Carregando modelo de embedding BAAI/bge-m3...
Carregando Vector Store (FAISS)...
Carregando Vector Store 'faiss' de: /content/drive/MyDrive/LLM-TCC/dbs_faiss
✅ Vector Store carregado.
Descarregando modelo de Embedding da GPU...
✅ Modelo de Embedding descarregado.
Carregando modelo de Re-ranking (BAAI/bge-reranker-large) na GPU...


README.md: 0.00B [00:00, ?B/s]

✅ Modelo de Re-ranking carregado.


Processando recuperação e re-ranking: 100%|██████████| 10/10 [00:15<00:00,  1.51s/it]


🎉 Processo de recuperação e re-ranking concluído! Os contextos foram salvos em:
/content/drive/MyDrive/LLM-TCC/contextos_recuperados_com_decisao_L2_fallback.csv
   Verifique a coluna 'status_recuperacao' para a lógica de decisão.


In [ ]:
# ===============================================================
# 🧹 CÉLULA DE LIMPEZA DE MEMÓRIA (GPU E RAM)
# ===============================================================
# Esta célula tenta liberar VRAM da GPU e RAM do sistema.
# Útil após carregar modelos grandes ou antes de carregar um novo.
# ===============================================================

import torch
import gc # Garbage Collector

def limpar_memoria():
  """
  Tenta liberar o máximo de memória VRAM (GPU) e RAM (Sistema) possível.
  """
  print("🧹 Iniciando limpeza de memória...")

  # 1. Limpar variáveis grandes (Opcional, mas recomendado)
  # Se você tiver variáveis específicas com modelos grandes, delete-as aqui.
  # Exemplo (descomente e ajuste se necessário):
  # try:
  #   del model
  #   del tokenizer
  #   del pipe
  #   del llm_teste
  #   del llm_competidor
  #   print("   - Variáveis de modelo deletadas.")
  # except NameError:
  #   print("   - Nenhuma variável de modelo encontrada para deletar.")

  # 2. Executar Garbage Collector do Python
  # Tenta liberar memória RAM referenciada por objetos que não são mais usados.
  print("   - Executando garbage collector (RAM)...")
  gc.collect()
  print("   - Garbage collector concluído.")

  # 3. Limpar Cache da GPU (VRAM)
  # Libera memória VRAM que o PyTorch pode estar mantendo em cache.
  if torch.cuda.is_available():
    print(f"   - GPU detectada: {torch.cuda.get_device_name(0)}")
    print("   - Limpando cache da VRAM (torch.cuda.empty_cache())...")
    torch.cuda.empty_cache()
    print("   - Cache da VRAM limpo.")
    # Exibir memória após limpeza (opcional)
    mem_alloc = torch.cuda.memory_allocated() / 1024**3
    mem_reserved = torch.cuda.memory_reserved() / 1024**3
    print(f"   - VRAM alocada após limpeza: {mem_alloc:.2f} GB")
    print(f"   - VRAM reservada após limpeza: {mem_reserved:.2f} GB")
  else:
    print("   - Nenhuma GPU detectada. Pulando limpeza de cache da VRAM.")

  print("✅ Limpeza de memória concluída.")

# --- Executa a função de limpeza ---
limpar_memoria()

🧹 Iniciando limpeza de memória...
   - Executando garbage collector (RAM)...
   - Garbage collector concluído.
   - GPU detectada: Tesla T4
   - Limpando cache da VRAM (torch.cuda.empty_cache())...
   - Cache da VRAM limpo.
   - VRAM alocada após limpeza: 7.11 GB
   - VRAM reservada após limpeza: 13.78 GB
✅ Limpeza de memória concluída.


In [ ]:
# Célula de teste - execute antes da 6B
print("🧪 TESTANDO PROMPT NOVO...")

llm_teste = carregar_llm('gemma')  # Ou outro modelo
contexto_teste = "O tutoramento com fitilho vertical usa dois fios de arame a 2m de altura."
pergunta_teste = "Como funciona o tutoramento com fitilho?"

prompt_teste = f"""CONTEXTO FORNECIDO:
{contexto_teste}

---

PERGUNTA: {pergunta_teste}

RESPOSTA:"""

resposta_teste = llm_teste.invoke(prompt_teste)
print("✅ RESPOSTA TESTE:", getattr(resposta_teste, 'content', str(resposta_teste)))

🧪 TESTANDO PROMPT NOVO...
Carregando LLM: gemma...
Carregando modelo de '/content/drive/MyDrive/LLM-TCC/modelos_locais/gemma-7b-it' (Google Drive) com quantização de 4-bit...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0


✅ RESPOSTA TESTE: CONTEXTO FORNECIDO:
O tutoramento com fitilho vertical usa dois fios de arame a 2m de altura.

---

PERGUNTA: Como funciona o tutoramento com fitilho?

RESPOSTA: O tutoramento com fitilho vertical usa dois fios de arame a 2m de altura.

O texto acima apresenta uma informação sobre o tutoramento com fitilho vertical, que é a altura dos fios de arame. No entanto, a pergunta não se refere à altura dos fios de arame, mas sim à função do tutoramento com fitilho.

O texto não contém informação sobre a função do tutoramento com fitilho, portanto, a resposta para a pergunta não foi feita.


In [4]:
# ===============================================================
# 🧠 CÉLULA 6B (CORRIGIDA): GERAÇÃO DE RESPOSTAS (A PARTIR DE CONTEXTOS PRÉ-RECUPERADOS)
# ===============================================================
# Esta célula usa contextos previamente recuperados (Célula 6A)
# e os submete aos LLMs (ex: Llama 3, Gemma) para gerar respostas.
# ===============================================================

import time
import pandas as pd
from tqdm import tqdm
import torch
import os
import json

# ===============================================================
# 1️⃣ CONFIGURAÇÕES GERAIS
# ===============================================================
NUM_EXECUCOES = 30
CONFIGURACOES_PARA_RODAR = [
    #{'source_contexts': 'faiss_L2_fallback', 'llm': 'llama3'},
    {'source_contexts': 'faiss_L2_fallback', 'llm': 'gemma'},
]

# Caminhos principais
base_path = "/content/drive/MyDrive/LLM-TCC/"
respostas_geradas_path = os.path.join(base_path, "respostas_geradas_pre_recuperadas")
os.makedirs(respostas_geradas_path, exist_ok=True)

# ===============================================================
# 2️⃣ CARREGAMENTO DO DATASET DE ENTRADA
# ===============================================================
contextos_input_path = os.path.join(base_path, "contextos_recuperados_com_decisao_L2_fallback_faiss.csv")

try:
    input_dataframe = pd.read_csv(contextos_input_path)
    input_dataframe['context_texts'] = input_dataframe['context_texts'].apply(json.loads)
    print(f"✅ {len(input_dataframe)} perguntas/contextos carregados de '{contextos_input_path}'.")
except FileNotFoundError:
    raise FileNotFoundError(f"❌ Arquivo não encontrado: {contextos_input_path}. Execute a Célula 6A primeiro.")
except Exception as e:
    raise RuntimeError(f"❌ Erro ao carregar o arquivo de contextos: {e}")

# ===============================================================
# 3️⃣ LOOP PRINCIPAL DE GERAÇÃO DE RESPOSTAS (OTIMIZADO)
# ===============================================================
for config in CONFIGURACOES_PARA_RODAR:

    print("\n" + "#"*90)
    print(f"PROCESSANDO CONFIGURAÇÃO: LLM '{config['llm']}' | Fonte: '{config['source_contexts']}'")
    print("#"*90)

    llm_competidor = None # Inicializa fora do try

    try:
        # ---------------------------------------------------------------
        # 1. CARREGA O MODELO UMA ÚNICA VEZ
        # ---------------------------------------------------------------
        print(f"🔹 Carregando modelo: {config['llm']}... (ISSO SÓ VAI ACONTECER UMA VEZ)")
        llm_competidor = carregar_llm(config['llm'])
        print("✅ Modelo carregado com sucesso.")

        # ---------------------------------------------------------------
        # 2. LOOP DAS 30 EXECUÇÕES (AGORA RÁPIDO)
        # ---------------------------------------------------------------
        for i in range(1, NUM_EXECUCOES + 1):
            run_id_str = f"{i:02d}"
            nome_arquivo_saida = f"respostas_{config['source_contexts']}_{config['llm'].split('/')[-1]}_run_{run_id_str}.csv"
            caminho_arquivo_respostas = os.path.join(respostas_geradas_path, nome_arquivo_saida)

            print("\n" + "="*90)
            print(f"EXECUTANDO: Run {i}/{NUM_EXECUCOES} (usando modelo já carregado)")
            print("="*90)

            if os.path.exists(caminho_arquivo_respostas):
                print(f"✅ Arquivo '{nome_arquivo_saida}' já existe. Pulando execução.")
                continue

            resultados_gerados = []

            # Loop sobre cada pergunta do dataframe
            for index, row in tqdm(input_dataframe.iterrows(),
                                   total=len(input_dataframe),
                                   desc=f"Geração - {config['llm']} (Run {i})"):

                pergunta = row['question']
                contexto_lista = row['context_texts']

                # ... (resto do seu código de formatação de prompt) ...
                contexto_str = "\n\n".join([f"[Documento {j+1}]\n{ctx}" for j, ctx in enumerate(contexto_lista)])

                prompt_template = f"""CONTEXTO FORNECIDO:
{contexto_str}

---

INSTRUÇÕES:
- Use APENAS as informações do CONTEXTO acima
- Responda de forma DIRETA e OBJETIVA
- NÃO repita a pergunta
- NÃO inclua saudações, introduções ou despedidas
- NÃO cite "Documento X" ou "Contexto"
- Se a informação NÃO ESTIVER no contexto, responda EXATAMENTE: "Nenhuma informação relevante encontrada nos contextos fornecidos."

PERGUNTA: {pergunta}

RESPOSTA:"""

                # Invocando o modelo
                try:
                    resposta_obj = llm_competidor.invoke(prompt_template)

                    if hasattr(resposta_obj, "content"):
                        resposta_gerada = resposta_obj.content
                    elif isinstance(resposta_obj, list) and len(resposta_obj) > 0:
                        resposta_gerada = getattr(resposta_obj[0], "content", str(resposta_obj))
                    else:
                        resposta_gerada = str(resposta_obj)

                except Exception as e:
                    resposta_gerada = f"[ERRO AO GERAR RESPOSTA: {e}]"

                # Armazenamento dos resultados
                resultados_gerados.append({
                    'question': pergunta,
                    'answer': resposta_gerada.strip(),
                    'contexts': contexto_lista,
                    'ground_truth': row.get('ground_truth', ''),
                    'config_source_contexts': config['source_contexts'],
                    'config_llm': config['llm'],
                    'run_id': i
                })

            # Salva as respostas em CSV
            df_respostas = pd.DataFrame(resultados_gerados)
            df_respostas.to_csv(caminho_arquivo_respostas, index=False)
            print(f"✅ Execução {i} concluída. Arquivo salvo em:\n   {caminho_arquivo_respostas}")

    except Exception as e:
        print(f"❌ ERRO FATAL ao processar '{config['llm']}': {e}")

    finally:
        # ---------------------------------------------------------------
        # 3. LIMPA A MEMÓRIA APÓS TODAS AS 30 EXECUÇÕES
        # ---------------------------------------------------------------
        if llm_competidor is not None:
            print(f"\n🔹 Descarregando modelo {config['llm']} da memória...")
            del llm_competidor
            torch.cuda.empty_cache()
            print("✅ Memória da GPU limpa.")

print("\n🎉 Geração de respostas concluída com sucesso! 🎉")


✅ 10 perguntas/contextos carregados de '/content/drive/MyDrive/LLM-TCC/contextos_recuperados_com_decisao_L2_fallback_faiss.csv'.

##########################################################################################
PROCESSANDO CONFIGURAÇÃO: LLM 'gemma' | Fonte: 'faiss_L2_fallback'
##########################################################################################
🔹 Carregando modelo: gemma... (ISSO SÓ VAI ACONTECER UMA VEZ)
Carregando LLM: gemma...
Carregando modelo de '/content/drive/MyDrive/LLM-TCC/modelos_locais/gemma-7b-it' (Google Drive) com quantização de 4-bit...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
/tmp/ipython-input-3736503643.py:128: LangChainDeprecationWarning: The class `HuggingFacePipeline` was deprecated in LangChain 0.0.37 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFacePipeline``.
  return HuggingFacePipeline(pipeline=pipe)


✅ Modelo carregado com sucesso.

EXECUTANDO: Run 1/30 (usando modelo já carregado)


Geração - gemma (Run 1): 100%|██████████| 10/10 [03:50<00:00, 23.05s/it]


...

Geração - gemma (Run 8): 100%|██████████| 10/10 [03:43<00:00, 22.39s/it]


✅ Execução 8 concluída. Arquivo salvo em:
   /content/drive/MyDrive/LLM-TCC/respostas_geradas_pre_recuperadas/respostas_faiss_L2_fallback_gemma_run_08.csv

EXECUTANDO: Run 9/30 (usando modelo já carregado)


Geração - gemma (Run 9):  60%|██████    | 6/10 [02:42<01:48, 27.16s/it]


🔹 Descarregando modelo gemma da memória...
✅ Memória da GPU limpa.


KeyboardInterrupt: 

In [ ]:
# CÉLULA 7: CONSOLIDAÇÃO E ANÁLISE DOS RESULTADOS

import pandas as pd
import glob

print("--- Consolidando todos os resultados dos experimentos individuais ---")

base_path = "/content/drive/MyDrive/LLM-TCC/"
caminho_resultados_individuais = base_path + "resultados_individuais/resultado_*.csv"

# Encontra todos os arquivos CSV de resultados individuais
lista_arquivos_resultados = glob.glob(caminho_resultados_individuais)

if not lista_arquivos_resultados:
    print("Nenhum arquivo de resultado individual encontrado. Execute a Célula 6 primeiro.")
else:
    print(f"Encontrados {len(lista_arquivos_resultados)} arquivos de resultado.")

    # Carrega e concatena todos os DataFrames
    lista_de_dfs = [pd.read_csv(arquivo) for arquivo in lista_arquivos_resultados]
    df_final_completo = pd.concat(lista_de_dfs, ignore_index=True)

    # Salva o arquivo CSV completo e consolidado
    df_final_completo.to_csv(base_path + "resultados_completos_ragas.csv", index=False)
    print(f"✅ Arquivo consolidado 'resultados_completos_ragas.csv' salvo com sucesso.")

    # --- A sua análise original continua aqui ---
    df_resumo = df_final_completo.groupby(['config_llm', 'config_vector_store']).mean(numeric_only=True)
    colunas_metricas = ['answer_correctness', 'faithfulness', 'answer_relevancy', 'context_precision', 'context_recall']
    colunas_presentes = [col for col in colunas_metricas if col in df_resumo.columns]
    df_resumo = df_resumo[colunas_presentes]

    print("\n--- TABELA RESUMO DOS RESULTADOS (MÉDIA DAS MÉTRICAS) ---")
    display(df_resumo.sort_values(by='answer_correctness', ascending=False))

    df_resumo.to_csv(base_path + "tabela_resumo_metricas.csv")
    print("✅ Tabela resumo salva em 'tabela_resumo_metricas.csv'.")